In [1]:
import json
import collections
import itertools
import os

In [2]:
DATA_DIR = '../nogit/quaternion/no_uncomp_data/rzrxrz/3q_4g_4blk_data/'
GOOD_DATA_PATH = DATA_DIR + 'good_fidelity/'
BAD_DATA_PATH = DATA_DIR + 'poor_fidelity/'
CONFIG_PATH = DATA_DIR + 'config.json'

In [3]:
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

print(config)

{'config': 'nogit/quaternion/no_uncomp_data/3q_4g_4blk_data/config.json', 'qubit_range': [3], 'gate_range': [4], 'pqc_blocks': 1, 'gate_blocks': 4, 'figure_output': 'nogit/quaternion/no_uncomp_data/rzrxrz/', 'epochs': 3, 'num_data': 2500, 'num_test': 50, 'seed': '10000', 'gate_dist': {'h': 0.5, 'cx': 0.25, 'x': 0.25}, 'noise_dist': {'x_rad': 0.0314, 'z_rad': 0.0, 'delta_x': 0.0, 'delta_z': 0.0}, 'gpu': False, 'batch': 10, 'force': True, 'redo': False, 'mp_cores': 0, 'uncomp': False, 'qubits': [3], 'gates': [4], 'device': 'cpu'}


In [4]:
good_data = []
poor_data = []

for filename in os.listdir(GOOD_DATA_PATH):
    with open(GOOD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        good_data.append((token_dict['seed'], token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
        f.close()

for filename in os.listdir(BAD_DATA_PATH):
    with open(BAD_DATA_PATH + filename, 'r') as f:
        token_dict = json.load(f)
        poor_data.append((token_dict['seed'], token_dict['base_circuit_tokens'], token_dict['pqc_params'], token_dict['fidelity']))
        f.close()

print(f"Number of good data samples: {len(good_data)}")
print(f"Number of poor data samples: {len(poor_data)}")

Number of good data samples: 10001
Number of poor data samples: 0


In [5]:
s, c, p, f = good_data[0]
print(type(s))
print(type(c))
print(type(p))
print(type(f))

<class 'int'>
<class 'list'>
<class 'list'>
<class 'float'>


In [6]:
test_good_data = good_data
dup_idx = []
for i, (s1, c1, p1, f1) in enumerate(test_good_data):
    for j, (s2, c2, p2, f2) in enumerate(test_good_data[i + 1:]):
        if all(x == y for x, y in zip(c1, c2)):
            print(f"Duplicate found {i}: {s1} and {i+1+j} : {s2}")
            print(f"Fidelities are \t{f1:.4f} and\t{f2:.4f}")
            # print(f"PQC params are {p1} and {p2}")
            dup_idx.append((i, j+i+1))
            


Duplicate found 0: 3721 and 5498 : 8077
Fidelities are 	1.0000 and	1.0000
Duplicate found 1: 2833 and 8202 : 2057
Fidelities are 	1.0000 and	1.0000
Duplicate found 4: 6489 and 4229 : 2619
Fidelities are 	0.9998 and	0.9998
Duplicate found 4: 6489 and 8399 : 8227
Fidelities are 	0.9998 and	0.9998
Duplicate found 6: 683 and 4078 : 2208
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 4258 : 8428
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 5531 : 8098
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 6313 : 5166
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 6774 : 8810
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 7344 : 9535
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 7522 : 7432
Fidelities are 	1.0000 and	1.0000
Duplicate found 6: 683 and 7668 : 7189
Fidelities are 	1.0000 and	1.0000
Duplicate found 8: 9286 and 1203 : 6368
Fidelities are 	1.0000 and	1.0000
Duplicate found 8: 9286 and 1316 : 4354
Fideli

In [7]:
print(f"Total duplicates found: {len(dup_idx)}")


Total duplicates found: 9293


In [8]:
S1 = 39
S2 = 135

In [9]:
print(good_data[S1])

(5533, [['cx', [1, 2], []], ['cx', [0, 2], []], ['h', [1], []], ['cx', [1, 0], []]], [[[3.1374456882476807, 0.06382761895656586, -3.137526035308838], [3.136805772781372, 0.031802404671907425, 3.1152284145355225], [-3.1227591037750244, 0.06179656460881233, 3.1235225200653076]]], 0.9997867941856384)


In [10]:
print(good_data[S2])

(8643, [['h', [2], []], ['h', [2], []], ['cx', [1, 0], []], ['h', [2], []]], [[[-3.1415843963623047, 0.03140253946185112, 3.141584634780884], [-3.141592025756836, 0.03140253946185112, 3.141592264175415], [3.125882387161255, 0.06279182434082031, 3.1258881092071533]]], 1.0)


In [11]:
c1 = good_data[S1][1]
c2 = good_data[S2][1]

list(zip(c1, c2))   

[(['cx', [1, 2], []], ['h', [2], []]),
 (['cx', [0, 2], []], ['h', [2], []]),
 (['h', [1], []], ['cx', [1, 0], []]),
 (['cx', [1, 0], []], ['h', [2], []])]

In [12]:
import numpy as np
p1 = good_data[S1][2]
p2 = good_data[S2][2]

[print(*p, sep="\n", end="\n***\n") for p in np.round(list(zip(*p1, *p2)), 5)]

[ 3.13745  0.06383 -3.13753]
[-3.14158  0.0314   3.14158]
***
[3.13681 0.0318  3.11523]
[-3.14159  0.0314   3.14159]
***
[-3.12276  0.0618   3.12352]
[3.12588 0.06279 3.12589]
***


[None, None, None]

In [13]:
print(p1)

[[[3.1374456882476807, 0.06382761895656586, -3.137526035308838], [3.136805772781372, 0.031802404671907425, 3.1152284145355225], [-3.1227591037750244, 0.06179656460881233, 3.1235225200653076]]]


In [14]:
print(p2)

[[[-3.1415843963623047, 0.03140253946185112, 3.141584634780884], [-3.141592025756836, 0.03140253946185112, 3.141592264175415], [3.125882387161255, 0.06279182434082031, 3.1258881092071533]]]


In [15]:
all(x == y for x, y in zip(c1, c2))

False

In [16]:
for idx1, idx2 in dup_idx:
    pqc1 = good_data[idx1][2]
    pqc2 = good_data[idx2][2]
    for (index1, value1), (index2, value2) in zip(np.ndenumerate(pqc1), np.ndenumerate(pqc2)):
        if abs(value1 - value2) > 1e-3:
            cos1 = np.cos(value1)
            cos2 = np.cos(value2)
            print(f"Difference found at pos {index1}, {index2} between {idx1} and {idx2}: {value1:0.5f} vs {value2:0.5f} (cos: {cos1:0.5f} vs {cos2:0.5f})")

Difference found at pos (0, 0, 0), (0, 0, 0) between 0 and 5498: -3.14151 vs 3.14156 (cos: -1.00000 vs -1.00000)
Difference found at pos (0, 0, 2), (0, 0, 2) between 0 and 5498: 3.14151 vs -3.14156 (cos: -1.00000 vs -1.00000)
Difference found at pos (0, 2, 0), (0, 2, 0) between 1 and 8202: -3.14155 vs 3.14145 (cos: -1.00000 vs -1.00000)
Difference found at pos (0, 2, 2), (0, 2, 2) between 1 and 8202: 3.14155 vs -3.14144 (cos: -1.00000 vs -1.00000)
Difference found at pos (0, 0, 0), (0, 0, 0) between 4 and 4229: -3.13798 vs 3.14067 (cos: -0.99999 vs -1.00000)
Difference found at pos (0, 0, 1), (0, 0, 1) between 4 and 4229: 0.06295 vs 0.05952 (cos: 0.99802 vs 0.99823)
Difference found at pos (0, 0, 2), (0, 0, 2) between 4 and 4229: 3.13814 vs -3.14058 (cos: -0.99999 vs -1.00000)
Difference found at pos (0, 1, 0), (0, 1, 0) between 4 and 4229: -3.13769 vs 3.14054 (cos: -0.99999 vs -1.00000)
Difference found at pos (0, 1, 2), (0, 1, 2) between 4 and 4229: 3.13774 vs -3.14029 (cos: -0.99999